# 05 — Profiling and Bottleneck Analysis

This notebook analyzes the performance characteristics of the selective scan kernel using measured GPU benchmark data.

**What you will learn:**
1. Why selective scan is memory-bound
2. How to read a roofline chart
3. Model Bandwidth Utilization (MBU) as a kernel quality metric
4. How the fused Triton kernel compares to the PyTorch reference path


## Memory-Bound vs Compute-Bound

Every GPU kernel sits somewhere on the roofline:

- **Compute-bound:** The kernel does enough math per byte of data that the GPU's ALUs are the bottleneck. Example: large matrix multiplies.
- **Memory-bound:** The kernel reads/writes more data than it computes on, so memory bandwidth is the bottleneck. Example: elementwise ops, reductions, and *recurrent scans*.

**Selective scan is memory-bound** because for each element, the recurrence does $O(N)$ FLOPs (where $N$ is the state dimension, typically 16), but needs to read $u$, $\Delta$, $B$, $C$ and write $y$ — that's many bytes per element with relatively little arithmetic.

The key metric is **arithmetic intensity** (FLOPs/byte):
- If AI < ridge point: you are memory-bound
- If AI > ridge point: you are compute-bound


## Load Measured Benchmark Data

In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Load the GPU scan benchmark results
scan_path = Path("../benchmarks/results/scan_results.gpu.json")
if not scan_path.exists():
    scan_path = Path("benchmarks/results/scan_results.gpu.json")

data = json.loads(scan_path.read_text())

# Separate by backend
ref_rows = [r for r in data if r["name"] == "reference"]
fused_rows = [r for r in data if r["name"] == "fused" and not r["used_fallback"]]

print(f"Loaded {len(data)} benchmark rows")
print(f"  Reference: {len(ref_rows)} configs")
print(f"  Fused Triton: {len(fused_rows)} configs")
print(f"  Hardware: {data[0]['machine']}")

## Latency: Fused Triton vs PyTorch Reference

In [ ]:
# Filter to B=1 for clean length sweep
ref_b1 = sorted([r for r in ref_rows if r["batch"] == 1], key=lambda r: r["length"])
fused_b1 = sorted([r for r in fused_rows if r["batch"] == 1], key=lambda r: r["length"])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Latency comparison
lengths_ref = [r["length"] for r in ref_b1]
lengths_fused = [r["length"] for r in fused_b1]
lat_ref = [r["latency_ms_p50"] for r in ref_b1]
lat_fused = [r["latency_ms_p50"] for r in fused_b1]

axes[0].plot(lengths_ref, lat_ref, "o-", label="PyTorch reference", linewidth=2, color="#4C78A8")
axes[0].plot(lengths_fused, lat_fused, "s-", label="Fused Triton", linewidth=2, color="#54A24B")
axes[0].set_xlabel("Sequence length")
axes[0].set_ylabel("Latency p50 (ms)")
axes[0].set_title("Selective Scan Latency (B=1, D=768, N=16)")
axes[0].set_yscale("log")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Speedup
common_lengths = sorted(set(lengths_ref) & set(lengths_fused))
ref_dict = {r["length"]: r["latency_ms_p50"] for r in ref_b1}
fused_dict = {r["length"]: r["latency_ms_p50"] for r in fused_b1}
speedups = [ref_dict[l] / fused_dict[l] for l in common_lengths]

axes[1].bar(range(len(common_lengths)), speedups, color="#54A24B", alpha=0.8)
axes[1].set_xticks(range(len(common_lengths)))
axes[1].set_xticklabels([str(l) for l in common_lengths])
axes[1].set_xlabel("Sequence length")
axes[1].set_ylabel("Speedup (x)")
axes[1].set_title("Fused Triton Speedup over PyTorch Reference")
axes[1].grid(True, alpha=0.3, axis="y")

for i, s in enumerate(speedups):
    axes[1].text(i, s + 1, f"{s:.0f}x", ha="center", fontsize=9, fontweight="bold")

fig.tight_layout()
plt.show()

print(f"\nSpeedups: {', '.join(f'L={l}: {s:.0f}x' for l, s in zip(common_lengths, speedups))}")

## Bandwidth Analysis

The fused kernel's goal is to minimize memory traffic by keeping intermediates in registers/SRAM. Let us see how much bandwidth it actually achieves:


In [ ]:
# A10G specs
PEAK_BW_GB_S = 600.0  # A10G theoretical peak memory bandwidth
PEAK_TFLOPS = 31.2    # A10G FP32 peak

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Achieved bandwidth across configs
bw_fused = [r["achieved_bandwidth_gb_s"] for r in fused_b1]
axes[0].plot(lengths_fused, bw_fused, "s-", linewidth=2, color="#54A24B")
axes[0].axhline(y=PEAK_BW_GB_S, color="red", linestyle="--", alpha=0.5, label=f"Peak BW ({PEAK_BW_GB_S} GB/s)")
axes[0].set_xlabel("Sequence length")
axes[0].set_ylabel("Achieved bandwidth (GB/s)")
axes[0].set_title("Fused Kernel Bandwidth (B=1)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MBU across all fused configs
mbu_values = [r["achieved_bandwidth_gb_s"] / PEAK_BW_GB_S * 100 for r in fused_rows]
configs = [f"B={r['batch']}\nL={r['length']}" for r in fused_rows]

axes[1].bar(range(len(mbu_values)), mbu_values, color="#54A24B", alpha=0.8)
axes[1].set_xticks(range(len(mbu_values)))
axes[1].set_xticklabels(configs, fontsize=7, rotation=45)
axes[1].set_ylabel("MBU (%)")
axes[1].set_title("Model Bandwidth Utilization (% of A10G peak)")
axes[1].grid(True, alpha=0.3, axis="y")

fig.tight_layout()
plt.show()

avg_mbu = np.mean(mbu_values)
max_mbu = max(mbu_values)
print(f"\nMBU stats: avg={avg_mbu:.1f}%, max={max_mbu:.1f}%")
print(f"This is typical for a scan kernel - the recurrence has limited parallelism within each channel.")

## Roofline Analysis

The roofline model plots achievable performance against arithmetic intensity. Points below the roofline ceiling indicate room for optimization; points near the memory-bandwidth slope indicate memory-bound behavior.


In [ ]:
# Compute arithmetic intensity and throughput for each fused config
intensities = [r["arithmetic_intensity_flops_per_byte"] for r in fused_rows]
throughputs = [r["achieved_bandwidth_gb_s"] * r["arithmetic_intensity_flops_per_byte"] for r in fused_rows]

# Build roofline
ai_range = np.logspace(-2, 3, 500)
bw_ceiling = PEAK_BW_GB_S * ai_range
compute_ceiling = PEAK_TFLOPS * 1e3
roofline = np.minimum(bw_ceiling, compute_ceiling)
ridge_point = compute_ceiling / PEAK_BW_GB_S

fig, ax = plt.subplots(figsize=(9, 6))
ax.loglog(ai_range, roofline, "b-", linewidth=2, label="Roofline")
ax.scatter(intensities, throughputs, c="crimson", s=60, zorder=5, label="Fused scan kernel (measured)")
ax.axvline(x=ridge_point, color="gray", linestyle=":", alpha=0.5, label=f"Ridge point ({ridge_point:.1f})")

ax.set_xlabel("Arithmetic Intensity (FLOPs/byte)")
ax.set_ylabel("Performance (GFLOPs/s)")
ax.set_title(f"Selective Scan Roofline (A10G: {PEAK_TFLOPS} TFLOPS, {PEAK_BW_GB_S} GB/s)")
ax.legend()
ax.grid(True, which="both", linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

print(f"\nAll kernel points sit LEFT of the ridge point ({ridge_point:.1f}) => memory-bound")
print(f"This confirms that selective scan performance is limited by memory bandwidth, not compute.")
print(f"Fusion helps by reducing the number of memory round-trips, not by doing less math.")

## Batch Size Effect

More batches increase occupancy and help amortize launch overhead:


In [ ]:
# Group fused results by batch size at a fixed length
target_length = 256
fused_by_batch = sorted(
    [r for r in fused_rows if r["length"] == target_length],
    key=lambda r: r["batch"]
)

if fused_by_batch:
    batches = [r["batch"] for r in fused_by_batch]
    bw = [r["achieved_bandwidth_gb_s"] for r in fused_by_batch]
    lat = [r["latency_ms_p50"] for r in fused_by_batch]
    
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].plot(batches, bw, "s-", linewidth=2, color="#54A24B")
    axes[0].set_xlabel("Batch size")
    axes[0].set_ylabel("Achieved bandwidth (GB/s)")
    axes[0].set_title(f"Bandwidth vs Batch Size (L={target_length})")
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(batches, lat, "s-", linewidth=2, color="#F58518")
    axes[1].set_xlabel("Batch size")
    axes[1].set_ylabel("Latency p50 (ms)")
    axes[1].set_title(f"Latency vs Batch Size (L={target_length})")
    axes[1].grid(True, alpha=0.3)
    
    fig.tight_layout()
    plt.show()

## Key Takeaways

1. **Selective scan is memory-bound** — all measured points sit well left of the roofline ridge point
2. **The fused Triton kernel achieves 70-120x speedup** over the PyTorch reference by eliminating Python loop overhead and reducing memory round-trips
3. **MBU is modest** — typical for scan kernels where the recurrence limits within-channel parallelism
4. **Fusion helps because it reduces memory traffic**, not because it reduces compute
5. **Larger batch sizes improve bandwidth utilization** by increasing GPU occupancy

This is exactly the bottleneck that motivates Mamba's architecture: since decode is memory-bound, Mamba's O(1) state (vs Transformer's growing KV cache) becomes a significant advantage at long contexts.
